# S.M.A.R.T. Feature Selection

Analyze the available S.M.A.R.T. attributes in the training dataset and select a set of features for the initial machine learning model.

In [2]:
import pandas as pd

training_df = pd.read_csv("../data/processed/training_data.csv")

smart_df = training_df.filter(like='smart')

# Counts the number of null values in each column
missing_rows = smart_df.isna().sum()

total_rows = len(smart_df)

missing_percentage = (missing_rows / total_rows) * 100

very_low_missing = missing_percentage[missing_percentage <= 5]
low_missing = missing_percentage[(missing_percentage <= 20) & (missing_percentage > 5)]
moderate_missing = missing_percentage[(missing_percentage <= 50) & (missing_percentage > 20)]
high_missing = missing_percentage[(missing_percentage <= 80) & (missing_percentage > 50)]
very_high_missing = missing_percentage[(missing_percentage <= 100) & (missing_percentage > 80)]

well_covered_smart_df = smart_df[very_low_missing.index]

feature_variance = well_covered_smart_df.var()

print(len(very_low_missing), len(low_missing), len(moderate_missing), len(high_missing), len(very_high_missing))
print(feature_variance.sort_values())

28 0 8 32 118
smart_10_raw            0.000000e+00
smart_4_normalized      1.820031e-03
smart_12_normalized     2.669952e-02
smart_10_normalized     2.308461e+00
smart_192_normalized    1.260352e+01
smart_194_raw           2.405107e+01
smart_197_normalized    3.394422e+01
smart_7_normalized      6.015265e+01
smart_5_normalized      6.069792e+01
smart_198_normalized    9.955423e+01
smart_193_normalized    1.015451e+02
smart_1_normalized      1.237030e+02
smart_3_normalized      5.302784e+02
smart_12_raw            9.061000e+02
smart_9_normalized      1.232256e+03
smart_199_normalized    1.661892e+03
smart_194_normalized    2.266354e+03
smart_4_raw             2.872966e+03
smart_199_raw           5.100680e+04
smart_3_raw             1.332463e+07
smart_5_raw             1.876433e+07
smart_198_raw           9.475086e+07
smart_197_raw           9.712411e+07
smart_9_raw             3.650012e+08
smart_192_raw           3.247873e+11
smart_193_raw           3.261543e+11
smart_1_raw             

### Variance Analysis

The variance was calculated for the S.M.A.R.T. features to identify features with little or no variation. `smart_10_raw` has zero variance which means its observed values are constant and it does not provide any useful information for finding the differences between observations. The remaining well-covered features do show some variation.

In [ ]:
model_counts = training_df["model"].value_counts()
major_models = model_counts[model_counts >= 1000]

major_model_names = major_models.index
candidate_features = list(very_low_missing.index)

major_model_df = training_df[training_df["model"].isin(major_model_names)]
worst_missing_by_feature = {}

for model in major_model_names:
    model_df = major_model_df[major_model_df["model"] == model]

    model_missing_percentage = (model_df[candidate_features].isna().mean() * 100)

    print(f"\n{model}")
    print(model_missing_percentage.sort_values(ascending=False).to_string())

    for feature in candidate_features:
        current_missing = model_missing_percentage[feature]

        if feature not in worst_missing_by_feature:
            worst_missing_by_feature[feature] = current_missing
        elif current_missing > worst_missing_by_feature[feature]:
            worst_missing_by_feature[feature] = current_missing


worst_missing_by_feature = pd.Series(worst_missing_by_feature)

print("\nWorst missing percentage across major models:")
print(worst_missing_by_feature.sort_values(ascending=False).to_string())



TOSHIBA MG08ACA16TA
smart_1_normalized      0.0
smart_1_raw             0.0
smart_3_normalized      0.0
smart_3_raw             0.0
smart_4_normalized      0.0
smart_4_raw             0.0
smart_5_normalized      0.0
smart_5_raw             0.0
smart_7_normalized      0.0
smart_7_raw             0.0
smart_9_normalized      0.0
smart_9_raw             0.0
smart_10_normalized     0.0
smart_10_raw            0.0
smart_12_normalized     0.0
smart_12_raw            0.0
smart_192_normalized    0.0
smart_192_raw           0.0
smart_193_normalized    0.0
smart_193_raw           0.0
smart_194_normalized    0.0
smart_194_raw           0.0
smart_197_normalized    0.0
smart_197_raw           0.0
smart_198_normalized    0.0
smart_198_raw           0.0
smart_199_normalized    0.0
smart_199_raw           0.0

TOSHIBA MG07ACA14TA
smart_1_normalized      0.0
smart_1_raw             0.0
smart_3_normalized      0.0
smart_3_raw             0.0
smart_4_normalized      0.0
smart_4_raw             0.0
smart_

In [4]:
selected_features = list(very_low_missing.index)
selected_features.remove("smart_10_raw")

print("Number of selected features:", len(selected_features))
print(selected_features)

Number of selected features: 27
['smart_1_normalized', 'smart_1_raw', 'smart_3_normalized', 'smart_3_raw', 'smart_4_normalized', 'smart_4_raw', 'smart_5_normalized', 'smart_5_raw', 'smart_7_normalized', 'smart_7_raw', 'smart_9_normalized', 'smart_9_raw', 'smart_10_normalized', 'smart_12_normalized', 'smart_12_raw', 'smart_192_normalized', 'smart_192_raw', 'smart_193_normalized', 'smart_193_raw', 'smart_194_normalized', 'smart_194_raw', 'smart_197_normalized', 'smart_197_raw', 'smart_198_normalized', 'smart_198_raw', 'smart_199_normalized', 'smart_199_raw']


### Initial Feature Selection

The original dataset contains 186 S.M.A.R.T. features with very different levels of data availability. The missing value analysis showed 28 features with 5% or less missing data across the training dataset.

The variance was then examined for these well-covered features. `smart_10_raw` had zero variance and was removed because a constant feature does not provide any useful information for finding the differences between observations.

The feature availability was also examined across the major drive models to check whether the selected features were consistently reported across different hardware models.

The initial feature set contains 27 S.M.A.R.T. features. The first model will use multiple drive models instead of being restricted to a single model. Model-specific performance can be evaluated later to determine whether or not separate models are necessary.